# Bring Your Own Explainer (BYOE) example

This example demonstrates how to run your custom explainer in order to interpret the model.

Techniques and methodologies used by H2O Sonar for model interpretation can be extended with recipes. Bring your own explainer (BYOE) feature allows you to use your explainers in combination with or in place of H2O Sonar built-in explainers (all H2O Sonar explainers are in fact BYOEs). This lets you extend the capabilities of H2O Sonar explainers and out-of-the-box interpretation techniques.

In [1]:
import sys
import os

import pandas
import webbrowser
from sklearn.ensemble import GradientBoostingClassifier
from h2o_sonar import interpret
from h2o_sonar.lib.api import models

# Load BYOE explainer

Load BYOE explainer to Python runtime so that it might be imported and subsequently run.

In [2]:
byoe_path = os.getcwd() + "/byoe/examples"
sys.path.append(os.path.abspath(byoe_path))

In [3]:
from example_metadata_explainer import ExampleMetaAndAttrsExplainer

# Register BYOE explainer

**Register** BYOE explainer in H2O Sonar runtime.

In [4]:
%%capture
explainer_id: str = interpret.register_explainer(
    explainer_class=ExampleMetaAndAttrsExplainer,
)

In [5]:
interpret.describe_explainer(ExampleMetaAndAttrsExplainer)

{'id': 'example_metadata_explainer.ExampleMetaAndAttrsExplainer',
 'name': 'ExampleMetaAndAttrsExplainer',
 'display_name': 'Example Explainer Metadata and Attributes',
 'tagline': 'ExampleMetaAndAttrsExplainer.',
 'description': 'This explainer example prints explainer metadata, instance attributes and setup() explainer parameters.',
 'brief_description': 'ExampleMetaAndAttrsExplainer.',
 'model_types': [],
 'can_explain': ['regression', 'binomial', 'multinomial'],
 'explanation_scopes': ['global_scope'],
 'explanations': [{'explanation_type': 'global-work-dir-archive',
   'name': 'WorkDirArchiveExplanation',
   'category': '',
   'scope': 'global',
   'has_local': '',
   'formats': []}],
 'keywords': [],
 'parameters': [],
 'metrics_meta': []}

# Run Interpretation

In [6]:
# prepare dataset and model to be explained

# dataset
target_col = "default payment next month"
df = pandas.read_csv("../../data/predictive/creditcard.csv")
(X, y) = df.drop(target_col, axis=1), df[target_col]

# scikit-learn model
gradient_booster = GradientBoostingClassifier(learning_rate=0.1)
gradient_booster.fit(X, y)

# explainable model
model = models.ModelApi().create_model(
    target_col=target_col, 
    model_src=gradient_booster, 
    used_features=X.columns.to_list()
)

results_location = "../../results"

In [7]:
# run the interpretation with BYOE explainer
interpretation = interpret.run_interpretation(
    dataset=df,
    model=model,
    target_col=target_col,
    results_location=results_location,
    # run the BYOE explainer only
    explainers=[explainer_id],
)

In [8]:
# retrieve the result
result = interpretation.get_explainer_result(explainer_id)

In [9]:
# summary
result.summary()

{'id': 'example_metadata_explainer.ExampleMetaAndAttrsExplainer',
 'name': 'ExampleMetaAndAttrsExplainer',
 'display_name': 'Example Explainer Metadata and Attributes',
 'tagline': 'ExampleMetaAndAttrsExplainer.',
 'description': 'This explainer example prints explainer metadata, instance attributes and setup() explainer parameters.',
 'brief_description': 'ExampleMetaAndAttrsExplainer.',
 'model_types': [],
 'can_explain': ['regression', 'binomial', 'multinomial'],
 'explanation_scopes': ['global_scope'],
 'explanations': [{'explanation_type': 'global-work-dir-archive',
   'name': 'ExampleMetaAndAttrsExplainer',
   'category': 'Demo',
   'scope': 'global',
   'has_local': None,
   'formats': ['application/zip']}],
 'keywords': [],
 'parameters': [],
 'metrics_meta': []}

In [10]:
# save the explainer log
result.log(path="./demo.log")

In [11]:
# open interpretation HTML report in web browser
print(f"Opening the result: {interpretation.result.get_html_report_location()}")
webbrowser.open(interpretation.result.get_html_report_location())

Opening the result: /home/dvorka/h/mli/git/h2o-sonar-FLOSS/results/h2o-sonar/mli_experiment_c95524c4-0e2a-4e3f-b033-927356f012a8/interpretation.html


True

In [12]:
!tree ../../results

../../results
├── h2o-sonar
│   ├── mli_experiment_93bb2f77-e2cf-472c-845d-b65dc699f0db
│   │   ├── explainer_example_metadata_explainer_ExampleMetaAndAttrsExplainer_9a617dc2-a3a6-4fb0-a284-d8b82f6d2d28
│   │   │   ├── global_work_dir_archive
│   │   │   │   ├── application_zip
│   │   │   │   │   └── explanation.zip
│   │   │   │   └── application_zip.meta
│   │   │   ├── insights
│   │   │   │   └── insights_and_actions.json
│   │   │   ├── log
│   │   │   │   └── explainer_run_9a617dc2-a3a6-4fb0-a284-d8b82f6d2d28.log
│   │   │   ├── problems
│   │   │   │   └── problems_and_actions.json
│   │   │   ├── result_descriptor.json
│   │   │   └── work
│   │   ├── interpretation.json
│   │   └── tmp
│   └── mli_experiment_c95524c4-0e2a-4e3f-b033-927356f012a8
│       ├── explainer_example_metadata_explainer_ExampleMetaAndAttrsExplainer_656eb7db-f250-4646-a27b-f0650213a324
│       │   ├── global_work_dir_archive
│       │   │   ├── application_zip
│       │   │   │   └── explanation.zip
│   

In [13]:
# save the explainer data
result.zip(file_path="./demo-archive.zip")

In [14]:
!unzip -l demo-archive.zip

Archive:  demo-archive.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
      978  2026-01-29 15:23   explainer_example_metadata_explainer_ExampleMetaAndAttrsExplainer_656eb7db-f250-4646-a27b-f0650213a324/result_descriptor.json
        2  2026-01-29 15:23   explainer_example_metadata_explainer_ExampleMetaAndAttrsExplainer_656eb7db-f250-4646-a27b-f0650213a324/problems/problems_and_actions.json
      142  2026-01-29 15:23   explainer_example_metadata_explainer_ExampleMetaAndAttrsExplainer_656eb7db-f250-4646-a27b-f0650213a324/global_work_dir_archive/application_zip.meta
       22  2026-01-29 15:23   explainer_example_metadata_explainer_ExampleMetaAndAttrsExplainer_656eb7db-f250-4646-a27b-f0650213a324/global_work_dir_archive/application_zip/explanation.zip
        0  2026-01-29 15:23   explainer_example_metadata_explainer_ExampleMetaAndAttrsExplainer_656eb7db-f250-4646-a27b-f0650213a324/log/explainer_run_656eb7db-f250-4646-a27b-f0650213a324.log
        2  2026-01-2